<a href="https://colab.research.google.com/github/divyanshuraj25/Day_19_Prompt_Engineering_Reliable_LLM_Outputs/blob/main/Day_19_Prompt_Engineering_Reliable_LLM_Outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# DAY 19 — Prompt Engineering for Reliable LLM Outputs

## RAG Question Answering Prompt Evaluation

This notebook evaluates five prompt engineering techniques on the same
10-question RAG dataset and compares their impact on answer quality,
accuracy, and format consistency.

In [2]:
!pip install -q google-genai pandas

In [3]:
from google import genai
import os
import getpass

# Enter your Gemini API key securely
API_KEY = getpass.getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-2.5-flash"

print("Gemini API client initialized successfully!")
print("Model:", MODEL_NAME)

Enter your Gemini API Key: ··········
Gemini API client initialized successfully!
Model: gemini-2.5-flash


In [4]:
# Day 19 — Evaluation Dataset

questions = [
    {
        "id": 1,
        "question": "What is artificial intelligence?",
        "expected_answer": "Artificial intelligence is the field of creating machines or systems that can perform tasks that normally require human intelligence."
    },
    {
        "id": 2,
        "question": "What is machine learning?",
        "expected_answer": "Machine learning is a branch of AI in which computers learn patterns from data and use them to make predictions or decisions."
    },
    {
        "id": 3,
        "question": "What is deep learning?",
        "expected_answer": "Deep learning is a type of machine learning that uses neural networks with multiple layers to learn complex patterns."
    },
    {
        "id": 4,
        "question": "What is natural language processing?",
        "expected_answer": "Natural language processing is a field of AI that enables computers to understand, process, and generate human language."
    },
    {
        "id": 5,
        "question": "What is a neural network?",
        "expected_answer": "A neural network is a machine learning model inspired by the structure of the human brain and consists of interconnected computational units."
    },
    {
        "id": 6,
        "question": "What is supervised learning?",
        "expected_answer": "Supervised learning is a machine learning approach where a model learns from labeled training data."
    },
    {
        "id": 7,
        "question": "What is unsupervised learning?",
        "expected_answer": "Unsupervised learning is a machine learning approach where a model finds patterns or structures in unlabeled data."
    },
    {
        "id": 8,
        "question": "What is reinforcement learning?",
        "expected_answer": "Reinforcement learning is a machine learning approach where an agent learns by interacting with an environment and receiving rewards or penalties."
    },
    {
        "id": 9,
        "question": "What is an embedding?",
        "expected_answer": "An embedding is a numerical vector representation of data such as text that captures semantic meaning."
    },
    {
        "id": 10,
        "question": "What is Retrieval-Augmented Generation?",
        "expected_answer": "Retrieval-Augmented Generation combines information retrieval with text generation so an LLM can generate answers using retrieved external context."
    }
]

print(f"Total evaluation questions: {len(questions)}")

Total evaluation questions: 10


In [5]:
# DAY 19 — Baseline Prompt

BASELINE_PROMPT = """
Answer the following question accurately and clearly.

Question:
{question}

Answer:
"""

print("Baseline prompt created successfully!")

Baseline prompt created successfully!


In [6]:
# DAY 19 — Generate Baseline Responses

baseline_results = []

for item in questions:
    prompt = BASELINE_PROMPT.format(question=item["question"])

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        answer = response.text.strip()

        baseline_results.append({
            "id": item["id"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "answer": answer,
            "status": "Success"
        })

        print(f"Q{item['id']} completed")

    except Exception as e:
        baseline_results.append({
            "id": item["id"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "answer": "",
            "status": f"API Error: {str(e)}"
        })

        print(f"Q{item['id']} API Error")

print("\nBaseline evaluation completed!")
print("Total results:", len(baseline_results))

Q1 API Error
Q2 API Error
Q3 API Error
Q4 API Error
Q5 API Error
Q6 API Error
Q7 API Error
Q8 API Error
Q9 API Error
Q10 API Error

Baseline evaluation completed!
Total results: 10


In [7]:
import pandas as pd

baseline_df = pd.DataFrame(baseline_results)

display(
    baseline_df[
        ["id", "question", "answer", "status"]
    ]
)

,id,question,answer,status
0,1,What is artificial intelligence?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
1,2,What is machine learning?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
2,3,What is deep learning?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
3,4,What is natural language processing?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
4,5,What is a neural network?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
5,6,What is supervised learning?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
6,7,What is unsupervised learning?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
7,8,What is reinforcement learning?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
8,9,What is an embedding?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...
9,10,What is Retrieval-Augmented Generation?,,API Error: 404 NOT_FOUND. {'error': {'code': 4...


In [8]:
# DAY 19 — Baseline Quality Evaluation

EVALUATOR_PROMPT = """
You are an objective evaluator.

Evaluate the generated answer against the expected answer.

Give scores from 1 to 5 for:

1. Accuracy — Is the answer factually correct?
2. Relevance — Does it directly answer the question?
3. Format Consistency — Is it clear, concise, and properly structured?

Return ONLY valid JSON in this exact format:

{
  "accuracy": 1,
  "relevance": 1,
  "format_consistency": 1
}

Question:
{question}

Expected Answer:
{expected_answer}

Generated Answer:
{answer}
"""

print("Evaluator prompt created successfully!")

Evaluator prompt created successfully!


In [9]:
import json
import time

def evaluate_answer(question, expected_answer, answer):
    prompt = EVALUATOR_PROMPT.format(
        question=question,
        expected_answer=expected_answer,
        answer=answer
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        text = response.text.strip()

        # Remove markdown code fences if Gemini adds them
        text = text.replace("```json", "").replace("```", "").strip()

        scores = json.loads(text)

        return {
            "accuracy": int(scores["accuracy"]),
            "relevance": int(scores["relevance"]),
            "format_consistency": int(scores["format_consistency"])
        }

    except Exception as e:
        return {
            "accuracy": None,
            "relevance": None,
            "format_consistency": None
        }


baseline_scores = []

for item in baseline_results:

    if item["status"] != "Success":
        baseline_scores.append({
            "id": item["id"],
            "accuracy": None,
            "relevance": None,
            "format_consistency": None
        })
        continue

    scores = evaluate_answer(
        item["question"],
        item["expected_answer"],
        item["answer"]
    )

    baseline_scores.append({
        "id": item["id"],
        **scores
    })

    print(f"Q{item['id']} evaluated")

    # Small delay to reduce API pressure
    time.sleep(1)

print("\nBaseline scoring completed!")


Baseline scoring completed!


In [10]:
# DAY 19 — Calculate Baseline Average Scores

scores_df = pd.DataFrame(baseline_scores)

baseline_accuracy = scores_df["accuracy"].mean()
baseline_relevance = scores_df["relevance"].mean()
baseline_format = scores_df["format_consistency"].mean()

baseline_overall = (
    baseline_accuracy +
    baseline_relevance +
    baseline_format
) / 3

print("=== BASELINE SCORES ===")
print(f"Accuracy:            {baseline_accuracy:.2f} / 5")
print(f"Relevance:           {baseline_relevance:.2f} / 5")
print(f"Format Consistency:  {baseline_format:.2f} / 5")
print(f"Overall Score:       {baseline_overall:.2f} / 5")

=== BASELINE SCORES ===
Accuracy:            nan / 5
Relevance:           nan / 5
Format Consistency:  nan / 5
Overall Score:       nan / 5


In [11]:
# DAY 19 — Technique 1: Role Assignment

ROLE_ASSIGNMENT_PROMPT = """
You are an expert AI and machine learning educator.

Answer the following question accurately, clearly, and directly.
Use simple language and avoid unnecessary information.

Question:
{question}

Answer:
"""

print("Role Assignment prompt created successfully!")

Role Assignment prompt created successfully!


In [12]:
# DAY 19 — Generate Role Assignment Responses

role_results = []

for item in questions:
    prompt = ROLE_ASSIGNMENT_PROMPT.format(
        question=item["question"]
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        answer = response.text.strip()

        role_results.append({
            "id": item["id"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "answer": answer,
            "status": "Success"
        })

        print(f"Q{item['id']} completed")

    except Exception as e:
        role_results.append({
            "id": item["id"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "answer": "",
            "status": f"API Error: {str(e)}"
        })

        print(f"Q{item['id']} API Error")

    time.sleep(1)

print("\nRole Assignment evaluation completed!")
print("Total results:", len(role_results))

Q1 API Error
Q2 API Error
Q3 API Error
Q4 API Error
Q5 API Error
Q6 API Error
Q7 API Error
Q8 API Error
Q9 API Error
Q10 API Error

Role Assignment evaluation completed!
Total results: 10


In [13]:
# Check the actual API error

try:
    test_response = client.models.generate_content(
        model=MODEL_NAME,
        contents="Say only: Role assignment test successful"
    )

    print(test_response.text)

except Exception as e:
    print("ACTUAL ERROR:")
    print(type(e).__name__)
    print(str(e))

ACTUAL ERROR:
ClientError
404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}


In [14]:
MODEL_NAME = "gemini-2.5-flash"

In [15]:
MODEL_NAME = "gemini-3.8-flash"

In [16]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say only: Gemini API connection successful"
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [17]:
import time

for attempt in range(3):
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents="Say only: Gemini API connection successful"
        )

        print(response.text)
        break

    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {type(e).__name__}")

        if attempt < 2:
            wait_time = 10 * (2 ** attempt)
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print("Gemini service is currently unavailable. Try again later.")

Attempt 1 failed: ServerError
Retrying in 10 seconds...
Attempt 2 failed: ServerError
Retrying in 20 seconds...
Attempt 3 failed: ServerError
Gemini service is currently unavailable. Try again later.


In [18]:
# DAY 19 — Technique 2: Output Format Specification

FORMAT_SPECIFICATION_PROMPT = """
Answer the following question accurately.

Return your answer using exactly this format:

Definition:
<one clear definition>

Key Point:
<one important point>

Question:
{question}
"""

print("Output Format Specification prompt created successfully!")

Output Format Specification prompt created successfully!


In [19]:
# STEP 14 — Test Output Format Prompt

test_question = questions[0]["question"]

test_prompt = FORMAT_SPECIFICATION_PROMPT.format(
    question=test_question
)

try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=test_prompt
    )

    print(response.text)

except Exception as e:
    print("API Error:", type(e).__name__)
    print(str(e))

API Error: ServerError
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [20]:
# DAY 19 — Prompt Engineering Experiment Tracker

experiment_results = {
    "baseline": [],
    "role_assignment": [],
    "format_specification": [],
    "reasoning": [],
    "few_shot": [],
    "negative_constraints": []
}

print("Experiment tracker created!")
print("Techniques:", list(experiment_results.keys()))

Experiment tracker created!
Techniques: ['baseline', 'role_assignment', 'format_specification', 'reasoning', 'few_shot', 'negative_constraints']


In [21]:
# DAY 19 — Technique 3: Reasoning-based Prompting

REASONING_PROMPT = """
You are an expert AI and machine learning educator.

Analyze the question carefully before answering.
Identify the key concept required to answer the question.
Then provide a concise and accurate final answer.

Question:
{question}

Final Answer:
"""

print("Reasoning-based prompt created successfully!")

Reasoning-based prompt created successfully!


In [22]:
# DAY 19 — Technique 4: Few-Shot Prompting

FEW_SHOT_PROMPT = """
Example 1:

Question:
What is machine learning?

Answer:
Machine learning is a branch of artificial intelligence in which computers learn patterns from data and make predictions.

--------------------------------

Example 2:

Question:
What is deep learning?

Answer:
Deep learning is a type of machine learning that uses multi-layer neural networks.

--------------------------------

Now answer the following question:

Question:
{question}

Answer:
"""

print("Few-shot prompt created successfully!")

Few-shot prompt created successfully!


In [23]:
# DAY 19 — Technique 5: Negative Constraints

NEGATIVE_CONSTRAINTS_PROMPT = """
Answer the following question accurately and clearly.

Rules:
- Do not add irrelevant information.
- Do not make unsupported claims.
- Do not use unnecessary technical jargon.
- Do not repeat the question.
- Do not give multiple answers when one clear answer is sufficient.
- Keep the answer concise.

Question:
{question}

Answer:
"""

print("Negative Constraints prompt created successfully!")

Negative Constraints prompt created successfully!


In [24]:
# DAY 19 — Grounding System Prompt

GROUNDING_PROMPT = """
You are a grounded question-answering assistant.

Use ONLY the information provided in the context to answer the question.

Rules:
- Do not use outside knowledge.
- Do not invent or assume facts.
- If the answer is not present in the context, clearly say:
  "I don't have enough information in the provided context to answer this question."
- Keep the answer concise and directly relevant.

Context:
{context}

Question:
{question}

Answer:
"""

print("Grounding system prompt created successfully!")

Grounding system prompt created successfully!


In [25]:
# DAY 19 — Out-of-Context Questions

ooc_questions = [
    "Who is the current President of India?",
    "What is the capital of Australia?",
    "What is the boiling point of water at sea level?",
    "Who invented the telephone?",
    "What is the population of Japan?"
]

print("Out-of-context questions created!")
print("Total OOC questions:", len(ooc_questions))

Out-of-context questions created!
Total OOC questions: 5


In [26]:
# DAY 19 — Test Grounding with One OOC Question

test_context = """
Artificial intelligence is a field of computer science.
Machine learning is a branch of artificial intelligence.
Deep learning uses neural networks with multiple layers.
"""

test_question = ooc_questions[0]

grounded_prompt = GROUNDING_PROMPT.format(
    context=test_context,
    question=test_question
)

try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=grounded_prompt
    )

    print("Question:", test_question)
    print("\nModel Response:")
    print(response.text)

except Exception as e:
    print("API Error:", type(e).__name__)
    print(str(e))

Question: Who is the current President of India?

Model Response:
I don't have enough information in the provided context to answer this question.


In [27]:
# DAY 19 — Grounding Test on 5 OOC Questions

ooc_results = []

for i, question in enumerate(ooc_questions, start=1):

    grounded_prompt = GROUNDING_PROMPT.format(
        context=test_context,
        question=question
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=grounded_prompt
        )

        answer = response.text.strip()

        ooc_results.append({
            "id": i,
            "question": question,
            "response": answer,
            "grounded": "I don't have enough information" in answer
        })

        print(f"Q{i} completed")

    except Exception as e:
        ooc_results.append({
            "id": i,
            "question": question,
            "response": "",
            "grounded": False
        })

        print(f"Q{i} API Error")

print("\nOOC Grounding Test Completed!")

Q1 API Error
Q2 API Error
Q3 API Error
Q4 API Error
Q5 API Error

OOC Grounding Test Completed!


In [28]:
# DAY 19 — Prompt Technique Comparison Structure

technique_names = [
    "Baseline",
    "Role Assignment",
    "Output Format Specification",
    "Reasoning-based Prompting",
    "Few-shot Prompting",
    "Negative Constraints"
]

comparison_df = pd.DataFrame({
    "Technique": technique_names,
    "Average Accuracy": [None] * 6,
    "Average Relevance": [None] * 6,
    "Format Consistency": [None] * 6,
    "Overall Score": [None] * 6
})

display(comparison_df)

,Technique,Average Accuracy,Average Relevance,Format Consistency,Overall Score
0,Baseline,None,None,None,None
1,Role Assignment,None,None,None,None
2,Output Format Specification,None,None,None,None
3,Reasoning-based Prompting,None,None,None,None
4,Few-shot Prompting,None,None,None,None
5,Negative Constraints,None,None,None,None


In [29]:
# DAY 19 — Prompt Technique Comparison Structure

technique_names = [
    "Baseline",
    "Role Assignment",
    "Output Format Specification",
    "Reasoning-based Prompting",
    "Few-shot Prompting",
    "Negative Constraints"
]

comparison_df = pd.DataFrame({
    "Technique": technique_names,
    "Average Accuracy": [None] * 6,
    "Average Relevance": [None] * 6,
    "Format Consistency": [None] * 6,
    "Overall Score": [None] * 6
})

display(comparison_df)

,Technique,Average Accuracy,Average Relevance,Format Consistency,Overall Score
0,Baseline,None,None,None,None
1,Role Assignment,None,None,None,None
2,Output Format Specification,None,None,None,None
3,Reasoning-based Prompting,None,None,None,None
4,Few-shot Prompting,None,None,None,None
5,Negative Constraints,None,None,None,None


In [30]:
# DAY 19 — Experiment Configuration

experiment_config = {
    "task": "RAG Question Answering",
    "number_of_questions": 10,
    "evaluation_scale": "1-5",
    "techniques": [
        "Baseline",
        "Role Assignment",
        "Output Format Specification",
        "Reasoning-based Prompting",
        "Few-shot Prompting",
        "Negative Constraints"
    ],
    "grounding_test_questions": 5
}

print("=== DAY 19 EXPERIMENT CONFIGURATION ===")

for key, value in experiment_config.items():
    print(f"{key}: {value}")

=== DAY 19 EXPERIMENT CONFIGURATION ===
task: RAG Question Answering
number_of_questions: 10
evaluation_scale: 1-5
techniques: ['Baseline', 'Role Assignment', 'Output Format Specification', 'Reasoning-based Prompting', 'Few-shot Prompting', 'Negative Constraints']
grounding_test_questions: 5


In [31]:
# DAY 19 — Final Results Template

final_results = {
    "Baseline": {
        "accuracy": None,
        "relevance": None,
        "format_consistency": None,
        "overall": None
    },
    "Role Assignment": {
        "accuracy": None,
        "relevance": None,
        "format_consistency": None,
        "overall": None
    },
    "Output Format Specification": {
        "accuracy": None,
        "relevance": None,
        "format_consistency": None,
        "overall": None
    },
    "Reasoning-based Prompting": {
        "accuracy": None,
        "relevance": None,
        "format_consistency": None,
        "overall": None
    },
    "Few-shot Prompting": {
        "accuracy": None,
        "relevance": None,
        "format_consistency": None,
        "overall": None
    },
    "Negative Constraints": {
        "accuracy": None,
        "relevance": None,
        "format_consistency": None,
        "overall": None
    }
}

print("Final results template created successfully!")

Final results template created successfully!


In [32]:
# DAY 19 — Final Experiment Summary

summary = """
DAY 19 — PROMPT ENGINEERING FOR RELIABLE LLM OUTPUTS

Task:
RAG Question Answering

Evaluation:
10 questions
Scoring scale: 1-5

Prompt Engineering Techniques:
1. Baseline Prompt
2. Role Assignment
3. Output Format Specification
4. Reasoning-based Prompting
5. Few-shot Prompting
6. Negative Constraints

Grounding Evaluation:
5 Out-of-Context questions

Grounding Rule:
The model must answer only from the provided context.
If the information is unavailable, the model must clearly refuse
instead of using outside knowledge.

Experiment Limitation:
Some Gemini API requests returned temporary 503 UNAVAILABLE errors.
Therefore, unavailable scores are not fabricated and are left unreported.
"""

print(summary)


DAY 19 — PROMPT ENGINEERING FOR RELIABLE LLM OUTPUTS

Task:
RAG Question Answering

Evaluation:
10 questions
Scoring scale: 1-5

Prompt Engineering Techniques:
1. Baseline Prompt
2. Role Assignment
3. Output Format Specification
4. Reasoning-based Prompting
5. Few-shot Prompting
6. Negative Constraints

Grounding Evaluation:
5 Out-of-Context questions

Grounding Rule:
The model must answer only from the provided context.
If the information is unavailable, the model must clearly refuse
instead of using outside knowledge.

Experiment Limitation:
Some Gemini API requests returned temporary 503 UNAVAILABLE errors.
Therefore, unavailable scores are not fabricated and are left unreported.



In [33]:
# DAY 19 — Hypothesis and Conclusion

hypothesis = """
HYPOTHESIS

Prompt engineering techniques that provide clearer instructions,
structured output requirements, examples, or explicit constraints
are expected to improve the reliability and consistency of LLM outputs.

Among the tested techniques, few-shot prompting and output format
specification are expected to have a noticeable effect on consistency,
while negative constraints may help reduce irrelevant or unsupported content.
"""

conclusion = """
DAY 19 — FINAL CONCLUSION

This experiment investigated how different prompt engineering
techniques can influence the reliability of LLM responses.

Six prompt versions were designed:
1. Baseline
2. Role Assignment
3. Output Format Specification
4. Reasoning-based Prompting
5. Few-shot Prompting
6. Negative Constraints

A separate grounding prompt was also created to prevent the model
from answering questions using information outside the provided context.

The grounding test successfully demonstrated the desired refusal
behavior for an out-of-context question.

Some model requests experienced temporary 503 UNAVAILABLE errors.
Therefore, numerical comparisons between prompt techniques were not
fabricated. The experiment framework is ready for completion when
the model service is available.
"""

print(hypothesis)
print(conclusion)


HYPOTHESIS

Prompt engineering techniques that provide clearer instructions,
structured output requirements, examples, or explicit constraints
are expected to improve the reliability and consistency of LLM outputs.

Among the tested techniques, few-shot prompting and output format
specification are expected to have a noticeable effect on consistency,
while negative constraints may help reduce irrelevant or unsupported content.


DAY 19 — FINAL CONCLUSION

This experiment investigated how different prompt engineering
techniques can influence the reliability of LLM responses.

Six prompt versions were designed:
1. Baseline
2. Role Assignment
3. Output Format Specification
4. Reasoning-based Prompting
5. Few-shot Prompting
6. Negative Constraints

A separate grounding prompt was also created to prevent the model
from answering questions using information outside the provided context.

The grounding test successfully demonstrated the desired refusal
behavior for an out-of-context question